# Stage 0 — Raw data ingestion

Downloads the UCI **ElectricityLoadDiagrams20112014** dataset (370 clients, 15-min consumption, 2011-2014)
and lands it in S3 as partitioned Parquet — our "raw" zone.

Source: https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014
License: CC BY 4.0

**Cost note**: this step is I/O-bound, not compute-bound. Run it on a small/free-tier-eligible
instance (e.g. `ml.t3.medium` Studio notebook) or locally — no need to burn a training-grade instance here.

In [ ]:
%pip install -q pyarrow boto3 requests tqdm

In [ ]:
import io
import zipfile
from pathlib import Path

import boto3
import pandas as pd
import requests
from tqdm import tqdm

# --- Config ------------------------------------------------------------
DATA_URL = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
LOCAL_DIR = Path("data_raw")
LOCAL_DIR.mkdir(exist_ok=True)
ZIP_PATH = LOCAL_DIR / "electricity.zip"
TXT_NAME = "LD2011_2014.txt"

BUCKET = "<your-bucket>"
RAW_PREFIX = "ts-forecast-demo/raw/electricity"

s3 = boto3.client("s3")

In [ ]:
def download_file(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    """Streams the file to disk with a progress bar (dataset is ~250MB zipped)."""
    if dest.exists():
        print(f"{dest} already exists, skipping download")
        return
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                bar.update(len(chunk))


download_file(DATA_URL, ZIP_PATH)

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(LOCAL_DIR)

txt_path = LOCAL_DIR / TXT_NAME
assert txt_path.exists(), f"Expected {txt_path} after extraction"
print(txt_path, txt_path.stat().st_size / 1e6, "MB")

## Parse and reshape

Raw file: semicolon-delimited, decimal comma, one column per client (`MT_001` ... `MT_370`),
first column is the timestamp. We reshape wide -> long (`timestamp, client_id, kw`) which is the
natural tidy format for a multi-series forecasting problem, and partition-friendly for Parquet.

In [ ]:
df_wide = pd.read_csv(
    txt_path,
    sep=";",
    decimal=",",
    index_col=0,
    parse_dates=True,
)
df_wide.index.name = "timestamp"
print(df_wide.shape)  # ~140256 timestamps x 370 clients
df_wide.head()

In [ ]:
# Wide -> long. kW -> kWh per the dataset docs (values are kW per 15-min slot; /4 = kWh).
df_long = (
    df_wide
    .reset_index()
    .melt(id_vars="timestamp", var_name="client_id", value_name="kw")
)
df_long["kwh"] = df_long["kw"] / 4.0
df_long["year"] = df_long["timestamp"].dt.year
df_long = df_long.sort_values(["client_id", "timestamp"]).reset_index(drop=True)
df_long.head()

## Write partitioned Parquet to S3

Partitioning by `year` keeps individual files a manageable size and lets later Processing/Training
jobs prune irrelevant partitions (e.g. train only on 2012-2014, matching common benchmark practice
of dropping the mostly-inactive 2011 partition).

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(df_long, preserve_index=False)

local_out = LOCAL_DIR / "parquet"
pq.write_to_dataset(
    table,
    root_path=str(local_out),
    partition_cols=["year"],
)

print(sorted(p.name for p in local_out.iterdir()))

In [ ]:
def upload_dir_to_s3(local_dir: Path, bucket: str, prefix: str) -> None:
    files = list(local_dir.rglob("*.parquet"))
    for f in tqdm(files):
        key = f"{prefix}/{f.relative_to(local_dir)}"
        s3.upload_file(str(f), bucket, key)


upload_dir_to_s3(local_out, BUCKET, RAW_PREFIX)
print(f"Uploaded to s3://{BUCKET}/{RAW_PREFIX}/")

In [ ]:
# Sanity check: list what landed in S3
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=RAW_PREFIX)
for obj in resp.get("Contents", [])[:10]:
    print(obj["Key"], obj["Size"])